# 03 - Reducción de Dimensionalidad (PCA y t-SNE)
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. Fundamentos Matemáticos de la Reducción de Dimensionalidad
Dado que la ingeniería de características genera un espacio de alta dimensionalidad (múltiples curvas $\times$ múltiples fases ZOI), el análisis enfrenta el riesgo del ruido y la multicolinealidad.
- **Análisis de Componentes Principales (PCA)**: Técnica lineal ortogonal que proyecta el espacio de características en direcciones de máxima varianza (autovectores de la matriz de covarianza), permitiendo evaluar la contribución (*loadings*) de cada señal.
- **t-Distributed Stochastic Neighbor Embedding (t-SNE)**: Técnica no lineal probabilística para preservar la estructura de vecindad local en bajas dimensiones.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_style_clustering.data.loader import load_session, enable_cache
from f1_style_clustering.features.zoi import extract_circuit_zones
from f1_style_clustering.features.aggregators import build_session_feature_matrix
from f1_style_clustering.features.preprocessor import normalize_relative_to_mean, standardize_features
from f1_style_clustering.models.dimensionality import perform_pca, compute_pca_loadings, perform_tsne
from f1_style_clustering.visualization.cluster_plots import plot_pca_biplot

enable_cache('cache')
session = load_session(year=2025, grand_prix='Italian Grand Prix', session_type='R')
analysis_zones = extract_circuit_zones(session)
df_raw = build_session_feature_matrix(session, analysis_zones=analysis_zones)
df_norm, df_meta = normalize_relative_to_mean(df_raw)
df_standardized, _, _ = standardize_features(df_norm)
print(f'Datos estandarizados listos: {df_standardized.shape}')


## 2. Análisis de Componentes Principales (PCA)

In [ ]:
# Ajustar PCA con 2 componentes
df_pca, pca_model, pca_summary = perform_pca(df_standardized, n_components=2)
print(f"Varianza explicada por PC1: {pca_summary['explained_variance_ratio'][0]*100:.2f}%")
print(f"Varianza explicada por PC2: {pca_summary['explained_variance_ratio'][1]*100:.2f}%")
print(f"Varianza explicada acumulada: {pca_summary['cumulative_variance']*100:.2f}%")

display(df_pca.head(10))


## 3. Análisis de Cargas Factoriales (*Loadings*) y Biplot de PCA

In [ ]:
# Calcular matriz de loadings
df_loadings = compute_pca_loadings(pca_model, df_standardized.columns.tolist())
print('--- Top 5 Features con mayor peso en PC1 ---')
display(df_loadings['PC1'].abs().sort_values(ascending=False).head(5))

# Biplot interactivo
fig_biplot = plot_pca_biplot(df_pca, df_loadings, top_n_features=6)
plt.show()


## 4. Proyección No Lineal con t-SNE

In [ ]:
# Ajustar t-SNE
df_tsne, tsne_model = perform_tsne(df_standardized, n_components=2, perplexity=5.0)

plt.figure(figsize=(9, 7))
sns.scatterplot(data=df_tsne, x='TSNE1', y='TSNE2', s=160, color='#00d2be', edgecolor='black')
for drv in df_tsne.index:
    plt.annotate(drv, (df_tsne.loc[drv, 'TSNE1'] + 0.3, df_tsne.loc[drv, 'TSNE2'] + 0.3), fontsize=10, fontweight='bold')

plt.title('Proyección 2D t-SNE de Estilos de Conducción', fontsize=13, fontweight='bold')
plt.xlabel('t-SNE Dimensión 1')
plt.ylabel('t-SNE Dimensión 2')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()
